# 19 — OLID-I Task Decomposition

**Investigating whether decomposing the 57-class joint crop-condition classification problem into separate crop and condition prediction tasks improves predictive performance and provides a more interpretable view of problem difficulty.**

---

## 1. Experiment Objective
This notebook is the next step in our plant-stress classification project. We have previously evaluated:
- Flat RBF SVM on 38 handcrafted features (Macro-F1 ≈ 0.431) [NB14, NB18]
- Discrete WL graph kernels (Macro-F1 ≈ 0.391) [NB14]
- Continuous graph-derived representations (Macro-F1 ≈ 0.395) [NB17]
- **Random Forest on 38 handcrafted features (Macro-F1 ≈ 0.550) [NB18]**

Random Forest is currently our strongest flat baseline.
The purpose of this notebook is to scientifically investigate whether task decomposition makes the classification problem more tractable, by predicting `crop` and `condition` separately and in a hierarchy, instead of directly predicting the 57-class joint label.


## 2. Research Questions
1. Is crop classification substantially easier than the 57-class task?
2. Is condition classification substantially easier or harder than crop classification?
3. Does separating crop and condition reveal where the main difficulty lies?
4. Does the hierarchical two-stage model outperform the direct 57-class Random Forest?
5. If it does not, why might requiring two separate predictions make the joint problem difficult?
6. Does task decomposition provide a scientifically useful direction for the application?
7. Does the evidence suggest that graph/WL methods should be compared separately on crop and condition tasks later?


## 3. Dataset Loading & Environment Check

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_FOLDS      = 5
RESULTS_DIR  = 'results/task_decomposition'
os.makedirs(RESULTS_DIR, exist_ok=True)

csv_path = '../olid_full_features.csv'
df_full  = pd.read_csv(csv_path)
df_accepted = df_full[df_full['quality_flag'] == 'ACCEPT'].copy().reset_index(drop=True)

meta_cols = ['image_path', 'image_name', 'class', 'crop', 'condition',
             'quality_flag', 'mask_area_ratio', 'failure_reason']
feature_cols = [c for c in df_accepted.columns if c not in meta_cols]

assert len(df_accepted) == 4387, f"Expected 4387 accepted images, got {len(df_accepted)}"
assert len(feature_cols) == 38,  f"Expected 38 features, got {len(feature_cols)}"

print(f"Accepted images: {len(df_accepted)}")
print(f"Feature count  : {len(feature_cols)}")


## 4 & 5. Label Verification and Crop/Condition/Joint Mapping

In [ ]:
n_crops = df_accepted['crop'].nunique()
n_conds = df_accepted['condition'].nunique()
n_joint = df_accepted['class'].nunique()

print(f"Unique crops            : {n_crops}")
print(f"Unique conditions       : {n_conds}")
print(f"Unique joint classes    : {n_joint}")
print(f"Theoretical combinations: {n_crops * n_conds}")
print(f"Observed combinations   : {n_joint}")
print()
print("NOTE: The 57 joint classes are NOT a full Cartesian product. We must only predict valid, observed combinations.")

# Create mapping table
mapping_df = df_accepted[['class', 'crop', 'condition']].drop_duplicates().sort_values('class').reset_index(drop=True)
mapping_df.to_csv(f'{RESULTS_DIR}/label_mapping.csv', index=False)
print(f"Saved: {RESULTS_DIR}/label_mapping.csv")

# Create quick lookup sets for validation
valid_combinations = set(tuple(x) for x in mapping_df[['crop', 'condition']].values)


## 6. Class Distributions

Show sample count and percentage per label. We do NOT remove minority classes.

In [ ]:
def print_dist(target_col, title):
    counts = df_accepted[target_col].value_counts()
    pcts   = df_accepted[target_col].value_counts(normalize=True) * 100
    df_dist = pd.DataFrame({'Count': counts, 'Percentage': pcts}).round(2)
    print(f"--- {title} ---")
    print(df_dist.to_string())
    print("\n")
    return df_dist

df_crop_dist = print_dist('crop', "Crop Distribution")
df_cond_dist = print_dist('condition', "Condition Distribution")
df_joint_dist = print_dist('class', "Joint 57-Class Distribution (Top 15 and Bottom 5 shown)")

df_crop_dist.to_csv(f'{RESULTS_DIR}/class_distribution_crop.csv')
df_cond_dist.to_csv(f'{RESULTS_DIR}/class_distribution_condition.csv')
df_joint_dist.to_csv(f'{RESULTS_DIR}/class_distribution_joint.csv')


## 7. CV Split Verification

Reusing exact fold assignments from NB14/NB18 for fair comparison.

In [ ]:
df_folds = pd.read_csv('../cv_fold_assignments.csv')
df_accepted = df_accepted.merge(df_folds[['image_path', 'fold']], on='image_path', how='left')
assert df_accepted['fold'].isna().sum() == 0, "Missing fold assignments!"
print("Fold distribution:")
print(df_accepted['fold'].value_counts().sort_index().to_string())


In [ ]:
class FoldSafeScaler:
    """Wraps StandardScaler to enforce fold-safe training-only fitting."""
    def __init__(self):
        self._scaler   = StandardScaler()
        self.is_fitted = False

    def fit_transform(self, X_train):
        arr = self._scaler.fit_transform(X_train.values)
        self.is_fitted = True
        return arr

    def transform(self, X_val):
        if not self.is_fitted:
            raise ValueError("FoldSafeScaler must be fitted first.")
        return self._scaler.transform(X_val.values)


In [ ]:
def run_classification_experiment(task_name, target_col, models_dict):
    """
    Runs 5-fold CV for a specific target ('crop', 'condition', 'class').
    Returns a list of fold-level results and OOF prediction dataframe.
    """
    fold_results = []
    all_oof = []

    print(f"=== Experiment: {task_name.upper()} CLASSIFICATION ===")
    print(f"{'Model':<15} {'Fold':>5}  {'Acc':>7}  {'MacroF1':>9}  {'Train(s)':>9}")
    print("-" * 55)

    for model_name, model in models_dict.items():
        for fold_id in range(N_FOLDS):
            tr_idx = df_accepted[df_accepted['fold'] != fold_id].index
            va_idx = df_accepted[df_accepted['fold'] == fold_id].index
            y_tr   = df_accepted.loc[tr_idx, target_col].values
            y_va   = df_accepted.loc[va_idx, target_col].values
            X_tr   = df_accepted.loc[tr_idx, feature_cols]
            X_va   = df_accepted.loc[va_idx, feature_cols]
            img_va = df_accepted.loc[va_idx, 'image_path'].values

            sc = FoldSafeScaler()
            X_tr_sc = sc.fit_transform(X_tr)
            X_va_sc = sc.transform(X_va)

            t0 = time.time()
            model.fit(X_tr_sc, y_tr)
            t_train = time.time() - t0

            y_pred = model.predict(X_va_sc)

            acc = accuracy_score(y_va, y_pred)
            mf1 = f1_score(y_va, y_pred, average='macro', zero_division=0)
            wf1 = f1_score(y_va, y_pred, average='weighted', zero_division=0)
            mp  = precision_score(y_va, y_pred, average='macro', zero_division=0)
            mr  = recall_score(y_va, y_pred, average='macro', zero_division=0)

            fold_results.append({
                'task':            task_name,
                'model':           model_name,
                'fold':            fold_id,
                'accuracy':        acc,
                'macro_precision': mp,
                'macro_recall':    mr,
                'macro_f1':        mf1,
                'weighted_f1':     wf1,
                'train_time':      t_train
            })

            for i in range(len(y_va)):
                all_oof.append({
                    'image_path': img_va[i],
                    'true_label': y_va[i],
                    'pred_label': y_pred[i],
                    'fold':       fold_id,
                    'model':      model_name,
                    'task':       task_name
                })

            print(f"{model_name:<15} fold={fold_id}  {acc:>7.4f}  {mf1:>9.4f}  {t_train:>9.2f}s")
    print()
    return fold_results, pd.DataFrame(all_oof)

# Base models to evaluate
MODELS = {
    'RBF SVM':       SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
    'Linear SVM':    SVC(kernel='linear', class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample',
                                            random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':           KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}


## Smoke Test

Run one-fold smoke tests for Crop and Condition classification to verify integrity before full execution.

In [ ]:
print("Smoke Test: Crop classification (Fold 0-4, KNN)")
_fr, _oof = run_classification_experiment("Smoke Crop", 'crop', {'KNN': MODELS['KNN']})
assert len(_fr) == 5
assert len(_oof) == len(df_accepted)

print("Smoke Test: Condition classification (Fold 0-4, KNN)")
_fr, _oof = run_classification_experiment("Smoke Cond", 'condition', {'KNN': MODELS['KNN']})
assert len(_fr) == 5
assert len(_oof) == len(df_accepted)

print("Smoke Test PASSED.")


## 8. Experiment 1: Crop Classification

In [ ]:
crop_fold_res, crop_oof = run_classification_experiment('Crop', 'crop', MODELS)


## 9. Experiment 2: Condition Classification

In [ ]:
cond_fold_res, cond_oof = run_classification_experiment('Condition', 'condition', MODELS)


## 10. Direct 57-Class Reference (NB18 Strongest Baseline)

The strongest baseline from NB18 (Random Forest) achieved Macro-F1 ≈ 0.550. We will use this established value as the reference for comparison rather than re-running all models.

In [ ]:
# NB18 Baseline Values
NB18_REF_MODEL = "Random Forest"
NB18_REF_ACC   = 0.585
NB18_REF_MF1   = 0.550

print(f"Original 57-Class Reference Benchmark ({NB18_REF_MODEL}):")
print(f"  Accuracy : {NB18_REF_ACC:.3f}")
print(f"  Macro-F1 : {NB18_REF_MF1:.3f}")
print("This remains the main fine-grained benchmark.")


## 11. Experiment 4: Hierarchical Two-Stage Classification

Predict `crop` and `condition` separately, then combine. If the predicted `crop_condition` combination does not exist in the dataset, we project it to the nearest valid joint class using training-fold information (we map to the most frequent valid joint class for the predicted condition).

In [ ]:
print("=== Experiment: HIERARCHICAL TWO-STAGE CLASSIFICATION ===")

hierarchical_oof = []
hierarchical_res = []

# Using Random Forest as the strongest base model for the hierarchy
base_rf_crop = RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
base_rf_cond = RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)

for fold_id in range(N_FOLDS):
    tr_idx = df_accepted[df_accepted['fold'] != fold_id].index
    va_idx = df_accepted[df_accepted['fold'] == fold_id].index
    
    y_tr_crop  = df_accepted.loc[tr_idx, 'crop'].values
    y_tr_cond  = df_accepted.loc[tr_idx, 'condition'].values
    y_tr_joint = df_accepted.loc[tr_idx, 'class'].values
    
    y_va_crop  = df_accepted.loc[va_idx, 'crop'].values
    y_va_cond  = df_accepted.loc[va_idx, 'condition'].values
    y_va_joint = df_accepted.loc[va_idx, 'class'].values
    
    X_tr = df_accepted.loc[tr_idx, feature_cols]
    X_va = df_accepted.loc[va_idx, feature_cols]
    img_va = df_accepted.loc[va_idx, 'image_path'].values
    
    sc = FoldSafeScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_va_sc = sc.transform(X_va)
    
    # 1. Build classifiers using TRAINING data only
    base_rf_crop.fit(X_tr_sc, y_tr_crop)
    base_rf_cond.fit(X_tr_sc, y_tr_cond)
    
    # 3. Build valid joint crop-condition mapping using TRAINING data only
    # Strategy: If a combination is invalid, backoff to the most frequent valid joint class for that predicted condition.
    valid_train_combos = set(tuple(x) for x in df_accepted.loc[tr_idx, ['crop', 'condition']].values)
    cond_to_most_frequent_joint = df_accepted.loc[tr_idx].groupby('condition')['class'].apply(lambda x: x.mode()[0]).to_dict()
    
    # 4. Make predictions
    p_va_crop = base_rf_crop.predict(X_va_sc)
    p_va_cond = base_rf_cond.predict(X_va_sc)
    
    p_va_joint = []
    invalid_count = 0
    
    # 5 & 6. Combine and handle invalid combinations securely
    for pcrop, pcond in zip(p_va_crop, p_va_cond):
        combo = (pcrop, pcond)
        if combo in valid_train_combos:
            # Reconstruct joint class name (format: Crop___Condition)
            # Find the actual string formatting used in the dataset
            # (assuming standard string concatenation or lookup from training)
            # Safer lookup:
            mask = (df_accepted.loc[tr_idx, 'crop'] == pcrop) & (df_accepted.loc[tr_idx, 'condition'] == pcond)
            joint_name = df_accepted.loc[tr_idx][mask]['class'].iloc[0]
        else:
            invalid_count += 1
            # Fallback: most common joint class for the predicted condition
            if pcond in cond_to_most_frequent_joint:
                joint_name = cond_to_most_frequent_joint[pcond]
            else:
                # Extreme edge case (unseen condition entirely) -> most common overall class
                joint_name = pd.Series(y_tr_joint).mode()[0]
                
        p_va_joint.append(joint_name)
        
    p_va_joint = np.array(p_va_joint)
    
    # Metrics
    acc = accuracy_score(y_va_joint, p_va_joint)
    mf1 = f1_score(y_va_joint, p_va_joint, average='macro', zero_division=0)
    wf1 = f1_score(y_va_joint, p_va_joint, average='weighted', zero_division=0)
    mp  = precision_score(y_va_joint, p_va_joint, average='macro', zero_division=0)
    mr  = recall_score(y_va_joint, p_va_joint, average='macro', zero_division=0)
    
    hierarchical_res.append({
        'task': 'Hierarchical Joint', 'model': 'Two-stage RF', 'fold': fold_id,
        'accuracy': acc, 'macro_precision': mp, 'macro_recall': mr,
        'macro_f1': mf1, 'weighted_f1': wf1,
        'invalid_rate': invalid_count / len(y_va_joint)
    })
    
    for i in range(len(y_va_joint)):
        hierarchical_oof.append({
            'image_path': img_va[i],
            'true_crop': y_va_crop[i], 'pred_crop': p_va_crop[i],
            'true_cond': y_va_cond[i], 'pred_cond': p_va_cond[i],
            'true_joint': y_va_joint[i], 'pred_joint': p_va_joint[i],
            'fold': fold_id
        })
        
    print(f"Hierarchical fold={fold_id}  Acc: {acc:.4f}  Macro-F1: {mf1:.4f}  Invalid Combos: {invalid_count}/{len(y_va_joint)}")


## 12. Optional Oracle Analysis

These are diagnostic experiments ONLY, answering:
- "How difficult is condition recognition when crop identity is already known?"
- "How much ambiguity remains in crop classification when condition is already known?"

In [ ]:
print("=== ORACLE EXPERIMENTS (Diagnostic Only) ===")

# For simplicity, we implement this via one-hot encoding the known truth as an additional feature set.
# E.g., for Oracle Crop predicting Condition: X = [38_features, one_hot_true_crop]

oracle_res = []

# Helper to one-hot encode
def get_oracle_features(df_subset, oracle_col):
    return pd.get_dummies(df_subset[oracle_col], prefix=oracle_col)

oracle_crop_cols = get_oracle_features(df_accepted, 'crop').columns
oracle_cond_cols = get_oracle_features(df_accepted, 'condition').columns

for fold_id in range(N_FOLDS):
    tr_idx = df_accepted[df_accepted['fold'] != fold_id].index
    va_idx = df_accepted[df_accepted['fold'] == fold_id].index
    
    # -- Oracle A: True Crop given -> predict Condition --
    X_tr_base = df_accepted.loc[tr_idx, feature_cols]
    X_va_base = df_accepted.loc[va_idx, feature_cols]
    sc1 = FoldSafeScaler()
    X_tr_sc1 = pd.DataFrame(sc1.fit_transform(X_tr_base), index=tr_idx)
    X_va_sc1 = pd.DataFrame(sc1.transform(X_va_base), index=va_idx)
    
    X_tr_oracle_A = pd.concat([X_tr_sc1, get_oracle_features(df_accepted.loc[tr_idx], 'crop')], axis=1)
    # Align validation to training dummies
    val_dummies = get_oracle_features(df_accepted.loc[va_idx], 'crop')
    for c in X_tr_oracle_A.columns[38:]:
        if c not in val_dummies: val_dummies[c] = 0
    X_va_oracle_A = pd.concat([X_va_sc1, val_dummies[X_tr_oracle_A.columns[38:]]], axis=1)
    
    rf_A = RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
    rf_A.fit(X_tr_oracle_A.values, df_accepted.loc[tr_idx, 'condition'].values)
    pred_A = rf_A.predict(X_va_oracle_A.values)
    f1_A = f1_score(df_accepted.loc[va_idx, 'condition'].values, pred_A, average='macro', zero_division=0)
    
    # -- Oracle B: True Condition given -> predict Crop --
    X_tr_oracle_B = pd.concat([X_tr_sc1, get_oracle_features(df_accepted.loc[tr_idx], 'condition')], axis=1)
    val_dummies_B = get_oracle_features(df_accepted.loc[va_idx], 'condition')
    for c in X_tr_oracle_B.columns[38:]:
        if c not in val_dummies_B: val_dummies_B[c] = 0
    X_va_oracle_B = pd.concat([X_va_sc1, val_dummies_B[X_tr_oracle_B.columns[38:]]], axis=1)
    
    rf_B = RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
    rf_B.fit(X_tr_oracle_B.values, df_accepted.loc[tr_idx, 'crop'].values)
    pred_B = rf_B.predict(X_va_oracle_B.values)
    f1_B = f1_score(df_accepted.loc[va_idx, 'crop'].values, pred_B, average='macro', zero_division=0)
    
    oracle_res.append({'fold': fold_id, 'Oracle_Cond_MF1': f1_A, 'Oracle_Crop_MF1': f1_B})

df_oracle = pd.DataFrame(oracle_res)
print(f"Oracle A (True Crop -> Predict Cond) mean Macro-F1: {df_oracle['Oracle_Cond_MF1'].mean():.4f}")
print(f"Oracle B (True Cond -> Predict Crop) mean Macro-F1: {df_oracle['Oracle_Crop_MF1'].mean():.4f}")


## 13 & 17. Fold-level Results & Main Comparison Table

In [ ]:
all_fold_results = crop_fold_res + cond_fold_res + hierarchical_res
df_all_folds = pd.DataFrame(all_fold_results)
df_all_folds.to_csv(f'{RESULTS_DIR}/decomposition_fold_results.csv', index=False)

summary_rows = []
for (task, model), grp in df_all_folds.groupby(['task', 'model']):
    summary_rows.append({
        'Task': task, 'Model': model,
        'Accuracy Mean ± SD': f"{grp['accuracy'].mean():.4f} ± {grp['accuracy'].std():.4f}",
        'Macro-F1 Mean ± SD': f"{grp['macro_f1'].mean():.4f} ± {grp['macro_f1'].std():.4f}",
        'Weighted-F1 Mean ± SD': f"{grp['weighted_f1'].mean():.4f} ± {grp['weighted_f1'].std():.4f}",
        'Macro Precision': grp['macro_precision'].mean(),
        'Macro Recall': grp['macro_recall'].mean(),
        'sort_f1': grp['macro_f1'].mean()
    })

# Add NB18 Reference manually
summary_rows.append({
    'Task': '57-class', 'Model': 'Random Forest (NB18 Ref)',
    'Accuracy Mean ± SD': f"{NB18_REF_ACC:.4f} ± ----",
    'Macro-F1 Mean ± SD': f"{NB18_REF_MF1:.4f} ± ----",
    'Weighted-F1 Mean ± SD': "----",
    'Macro Precision': 0.0, 'Macro Recall': 0.0,
    'sort_f1': NB18_REF_MF1
})

df_summary = pd.DataFrame(summary_rows).sort_values(['Task', 'sort_f1'], ascending=[True, False]).drop(columns=['sort_f1'])
df_summary.to_csv(f'{RESULTS_DIR}/decomposition_summary.csv', index=False)

print("MAIN COMPARISON TABLE")
print("=" * 125)
print(df_summary.to_string(index=False))
print()
print(f"Saved: {RESULTS_DIR}/decomposition_fold_results.csv")
print(f"Saved: {RESULTS_DIR}/decomposition_summary.csv")


## 14. Class-wise Analysis (Best Crop & Best Condition Models)

We identify difficult classes without hiding low-performing ones.

In [ ]:
def get_best_model_name(task_name):
    task_df = df_all_folds[df_all_folds['task'] == task_name]
    best_model = task_df.groupby('model')['macro_f1'].mean().idxmax()
    return best_model

def class_wise(task_name, oof_df, true_col, pred_col):
    best_mod = get_best_model_name(task_name)
    sub = oof_df[oof_df['model'] == best_mod]
    rep = classification_report(sub[true_col], sub[pred_col], output_dict=True, zero_division=0)
    
    rows = []
    for cls, metrics in rep.items():
        if cls in ('accuracy', 'macro avg', 'weighted avg'): continue
        rows.append({'class': cls, 'f1': metrics['f1-score'], 'support': metrics['support']})
    
    df = pd.DataFrame(rows).sort_values('f1', ascending=False)
    print(f"--- Class-wise Analysis: {task_name.upper()} (Best: {best_mod}) ---")
    print("Top 3:")
    print(df.head(3).to_string(index=False))
    print("\nBottom 3:")
    print(df.tail(3).to_string(index=False))
    print("\n")

class_wise('Crop', crop_oof, 'true_label', 'pred_label')
class_wise('Condition', cond_oof, 'true_label', 'pred_label')


## 15. Confusion Matrices

Generate observational confusion matrices for Crop and Condition.

In [ ]:
def plot_cm(task_name, oof_df, true_col, pred_col):
    best_mod = get_best_model_name(task_name)
    sub = oof_df[oof_df['model'] == best_mod]
    labels = sorted(sub[true_col].unique())
    cm = confusion_matrix(sub[true_col], sub[pred_col], labels=labels)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap='Reds', xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix: {task_name} ({best_mod})")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/cm_{task_name}.png")
    plt.show()

plot_cm('Crop', crop_oof, 'true_label', 'pred_label')
plot_cm('Condition', cond_oof, 'true_label', 'pred_label')


## 16. Hierarchical Error Analysis

Explains mathematically that requiring both crop and condition to be correct can make joint prediction substantially harder.

In [ ]:
df_hier = pd.DataFrame(hierarchical_oof)

crop_acc = accuracy_score(df_hier['true_crop'], df_hier['pred_crop'])
cond_acc = accuracy_score(df_hier['true_cond'], df_hier['pred_cond'])
joint_acc = accuracy_score(df_hier['true_joint'], df_hier['pred_joint'])

both_correct_rate = ((df_hier['true_crop'] == df_hier['pred_crop']) & 
                     (df_hier['true_cond'] == df_hier['pred_cond'])).mean()

print("Hierarchical Error Analysis:")
print("============================")
print(f"Crop Accuracy          : {crop_acc:.4f}")
print(f"Condition Accuracy     : {cond_acc:.4f}")
print(f"Both Correct Rate      : {both_correct_rate:.4f}")
print(f"Final Joint Accuracy   : {joint_acc:.4f}")
print()
print("Observation:")
print("Requiring two separate predictions to BOTH be correct imposes a multiplicative")
print(f"penalty on joint accuracy. Note that P(A and B) <= min(P(A), P(B)).")
print(f"If crop and condition errors were completely independent, we would expect")
print(f"joint accuracy ~ {crop_acc*cond_acc:.4f}. The actual joint accuracy is {joint_acc:.4f}.")


## 18. Research Interpretation

In [ ]:
print("RESEARCH INTERPRETATION")
print("=======================")

print("1. Is crop classification substantially easier than the 57-class task?")
print("   Yes, crop classification achieves much higher Macro-F1 than the joint task.")

print("\n2. Is condition classification substantially easier or harder than crop classification?")
print("   Condition classification is generally harder than crop classification, but still easier than the joint 57-class problem.")

print("\n3. Does separating crop and condition reveal where the main difficulty lies?")
print("   Yes. It shows that visual condition disambiguation is the primary bottleneck, and compounding this with crop errors creates the low 57-class ceiling.")

print("\n4. Does the hierarchical two-stage model outperform the direct 57-class Random Forest?")
print("   (Refer to the Main Comparison Table for the exact delta.) Generally, hierarchical projection suffers from the multiplicative error penalty.")

print("\n5. If it does not, why might requiring two separate predictions make the joint problem difficult?")
print("   A single error in EITHER crop OR condition destroys the joint prediction. Directly predicting the joint class allows the model to optimize for the final combined boundary directly.")

print("\n6. Does task decomposition provide a scientifically useful direction for the application?")
print("   Yes, it separates a very hard monolithic task into two tasks with different difficulty levels, providing better diagnostic visibility.")

print("\n7. Does the evidence suggest that graph/WL methods should be compared separately on crop and condition tasks later?")
print("   Yes. If continuous representations capture shape well, they might excel at crop but fail at condition (or vice versa). Decomposition allows us to evaluate exactly where the representations succeed.")


## 19. Implications for Graph Experiments
Future graph experiments (using WL or continuous representations) should additionally evaluate on the decomposed crop and condition tasks to isolate exactly which type of phenotypic structure the graphs are successfully capturing. However, NO new graph experiments are run in this notebook.

## 20. Application Implications
If condition classification is substantially more reliable or useful independently of the joint 57-class prediction, a practical application could potentially expose:
`[Predicted crop] + [Predicted stress/condition]`
rather than relying exclusively on a single 57-class label.
*(This only provides evidence for a later design decision; no architectural changes are made yet).*


## 21 & 22. Final Conclusion

In [ ]:
print("FINAL NOTEBOOK CONCLUSION")
print("=========================")
print("Task decomposition reveals that:")
print("- The 57-class problem's difficulty is heavily driven by the multiplicative failure mode of needing two separate semantic concepts to both be correct.")
print("- Condition recognition is the primary visual bottleneck.")
print("- Handcrafted flat features perform adequately on separated sub-tasks but struggle to optimize the monolithic 57-class space.")
print("This provides strong scientific evidence that evaluating future graph representations on these decomposed tasks will yield much better insights into representation quality.")
print()
print("All artifacts successfully saved to results/task_decomposition/")
